## Exercise6 - MCP Researcher
   This consists of:
1. **email server**
2. **push server** 
3. **fetch server**
4. **brave server**
5. **input gaurdrail**
6. **output gaurdrail**
7. **Gradio interface**
8. **openai agent sdk**

In [16]:
# imports

import os
from datetime import datetime
from pathlib import Path
from typing import Tuple

import gradio as gr
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from mcp.server.fastmcp import FastMCP
from pydantic import BaseModel, Field
import sendgrid
from sendgrid.helpers.mail import Content, Email, Mail, To

from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner, Tool, gen_trace_id, trace
from agents.mcp import MCPServerStdio

In [17]:
load_dotenv(override=True)

True

In [18]:

def _find_mcp_servers_dir() -> Path:
    """Folder containing push_server.py and email_server.py (walk up from cwd)."""
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "push_server.py").exists() and (d / "email_server.py").exists():
            return d
    for d in [start, *start.parents]:
        if (d / "push_server.py").exists():
            return d
    return start


_MCP_DIR = _find_mcp_servers_dir()
push_mcp_server_params = [
    {"command": "uv", "args": ["run", str(_MCP_DIR / "push_server.py")]}]
email_mcp_server_params = [
    {"command": "uv", "args": ["run", str(_MCP_DIR / "email_server.py")]}]
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-brave-search"],
        "env": brave_env,
    },
]

researcher_mcp_servers = [
    MCPServerStdio(params, client_session_timeout_seconds=30)
    for params in researcher_mcp_server_params
]
push_mcp_servers = [
    MCPServerStdio(params, client_session_timeout_seconds=30)
    for params in push_mcp_server_params
]
email_mcp_servers = [
    MCPServerStdio(params, client_session_timeout_seconds=30)
    for params in email_mcp_server_params
]
mcp_servers = push_mcp_servers + email_mcp_servers + researcher_mcp_servers

async def get_researcher(servers) -> Agent:
    instructions = f"""You are a helpful assistant. You browse the internet to accomplish your instructions,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.

You have tools for push notifications and for email (SendGrid). Use the push tool only when the user clearly asks for a push notification.
Use the email tool send_final_output_email only when the user explicitly asks to receive an email or to have results emailed.
Do not send push or email without that explicit request. After sending, briefly confirm in your reply what you sent.

The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    return Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=servers,
    )


async def get_researcher_tool(servers) -> Tool:
    researcher = await get_researcher(servers)
    return researcher.as_tool(
        tool_name="Researcher",
        tool_description="This tool researches online for news and opportunities, \
either based on your specific request, \
or generally for notable news and opportunities. \
Describe what kind of research you're looking for.",
    )


# Set True to run a one-shot researcher call in this cell (uses fetch + Brave + Push MCP).
RUN_NOTEBOOK_DEMO = False

if RUN_NOTEBOOK_DEMO:
    research_question = (
        "What's the latest news on Artificial intelligence and send a push notification"
    )
    for server in mcp_servers:
        await server.connect()
    researcher = await get_researcher(mcp_servers)
    _tid = gen_trace_id()
    print(f"View trace: https://platform.openai.com/traces/trace?trace_id={_tid}")
    with trace("Researcher", trace_id=_tid):
        result = await Runner.run(researcher, research_question, max_turns=30)
    display(Markdown(result.final_output))

In [19]:
MAX_QUESTION_CHARS = 8000
MIN_QUESTION_CHARS = 3
MAX_OUTPUT_DISPLAY_CHARS = 50_000

# Optional: add lowercase substrings to block (e.g. exfiltration prompts). Empty = disabled.
BLOCKED_INPUT_SUBSTRINGS: Tuple[str, ...] = ()


def guardrail_input(text: str) -> Tuple[bool, str]:
    if text is None:
        return False, "Please enter a research question."
    cleaned = str(text).replace("\x00", "").strip()
    if len(cleaned) < MIN_QUESTION_CHARS:
        return False, f"Please enter at least {MIN_QUESTION_CHARS} non-space characters."
    if len(cleaned) > MAX_QUESTION_CHARS:
        return False, f"Input is too long (max {MAX_QUESTION_CHARS} characters)."
    lower = cleaned.lower()
    for needle in BLOCKED_INPUT_SUBSTRINGS:
        if needle and needle in lower:
            return False, "That request is not allowed."
    return True, cleaned


def guardrail_output(text: str) -> str:
    if text is None:
        return "_No output._"
    cleaned = str(text).replace("\x00", "")
    if len(cleaned) > MAX_OUTPUT_DISPLAY_CHARS:
        tail = f"\n\n_(Output truncated for display; max {MAX_OUTPUT_DISPLAY_CHARS} characters.)_"
        return cleaned[:MAX_OUTPUT_DISPLAY_CHARS] + tail
    return cleaned

In [20]:
# If you re-run the MCP setup cell after launching this UI, restart the kernel so connections stay in sync.

_setup_done = False
_gradio_researcher = None


def _tool_name_from_raw(raw) -> str:
    if raw is None:
        return "?"
    if isinstance(raw, dict):
        return str(raw.get("name", raw))[:120]
    return str(getattr(raw, "name", raw))[:120]


def _short_text(s: object, n: int = 500) -> str:
    t = str(s) if s is not None else ""
    if len(t) <= n:
        return t
    return t[: n - 3] + "..."


async def _ensure_mcp_for_ui():
    global _setup_done, _gradio_researcher
    if _setup_done:
        return _gradio_researcher
    for server in mcp_servers:
        await server.connect()
    _gradio_researcher = await get_researcher(mcp_servers)
    _setup_done = True
    return _gradio_researcher


def _render_trace_ui(
    trace_url: str,
    log_lines: list[str],
    partial_answer: str,
    *,
    final_text: str | None = None,
) -> str:
    parts = [
        f"**OpenAI trace:** [{trace_url}]({trace_url})",
        "",
        "### Live activity",
        *(f"- {line}" for line in log_lines),
    ]
    if final_text is not None:
        parts.extend(["", "### Final response", "", final_text])
    elif partial_answer:
        parts.extend(["", "### Answer (streaming)", "", partial_answer])
    return guardrail_output("\n".join(parts))


async def run_research_ui(user_question: str):
    ok, msg = guardrail_input(user_question)
    if not ok:
        yield guardrail_output(msg)
        return

    trace_id = gen_trace_id()
    trace_url = f"https://platform.openai.com/traces/trace?trace_id={trace_id}"
    print(f"View trace: {trace_url}")

    log_lines: list[str] = []
    partial_answer = ""

    researcher = await _ensure_mcp_for_ui()

    with trace("Researcher", trace_id=trace_id):
        stream = Runner.run_streamed(
            researcher,
            input=msg,
            max_turns=30,
        )
        async for event in stream.stream_events():
            if event.type == "raw_response_event":
                if isinstance(event.data, ResponseTextDeltaEvent):
                    partial_answer += event.data.delta
                    yield _render_trace_ui(trace_url, log_lines, partial_answer)
                continue
            if event.type == "agent_updated_stream_event":
                name = getattr(event.new_agent, "name", "?")
                log_lines.append(f"**Agent step:** `{name}`")
                yield _render_trace_ui(trace_url, log_lines, partial_answer)
            elif event.type == "run_item_stream_event":
                item = event.item
                itype = getattr(item, "type", None)
                if itype == "tool_call_item":
                    log_lines.append(
                        f"**Tool call:** `{_tool_name_from_raw(getattr(item, 'raw_item', None))}`"
                    )
                    yield _render_trace_ui(trace_url, log_lines, partial_answer)
                elif itype == "tool_call_output_item":
                    log_lines.append(
                        f"**Tool output:** {_short_text(getattr(item, 'output', ''), 600)}"
                    )
                    yield _render_trace_ui(trace_url, log_lines, partial_answer)
                elif itype == "message_output_item":
                    out = getattr(item, "output", None)
                    if out is not None:
                        partial_answer = str(out)
                        yield _render_trace_ui(trace_url, log_lines, partial_answer)

        final = stream.final_output

    fin = str(final) if final is not None else partial_answer
    yield _render_trace_ui(trace_url, log_lines, partial_answer, final_text=fin)


demo = gr.Interface(
    fn=run_research_ui,
    inputs=gr.Textbox(
        label="Research request",
        placeholder="e.g. Latest AI news and send me a push notification with one headline",
        lines=4,
    ),
    outputs=gr.Markdown(label="Researcher response"),
    title="MCP Researcher",
    description=(
        "Uses fetch + Brave Search, Pushover and SendGrid email "
    ),
    examples=[
        ["Summarize today's top tech headlines and push a short alert."],
        ["Research about the latest technological advancement and email me a summary and a push notification."],
        ["What are notable renewable energy investment themes this week?"],
    ],
    flagging_mode="never",
)

demo.queue()
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


View trace: https://platform.openai.com/traces/trace?trace_id=trace_727b0bdabc8743618f01cf8914dedfeb
